# CPython Object-Oriented Programming (OOP) Engine, Design Patterns, and Interview Guide

## 1. THEORY & MEMORY MODEL

### Python Object Memory Model
In CPython, everything is an object. An object is represented in C memory as a struct containing:
- A reference count (`ob_refcnt`) which handles deterministic memory deallocation.
- A pointer to a type object (`ob_type`) which defines the object's methods, attributes, and base classes.

### Object Creation Lifecycle
Creating an instance of a class involves two key stages:
1. `__new__(cls, *args, **kwargs)`: Allocates raw memory on the heap for the object instance, returning the newly created object.
2. `__init__(self, *args, **kwargs)`: Initializes the object's instance variables (attribute bindings) inside the object's dictionary (`__dict__`).

### Identity vs Equality
- Identity (`is`): Evaluates whether two references point to the exact same memory address (uses C pointers comparison).
- Equality (`==`): Evaluates whether the contents of the objects are equivalent (maps to the `__eq__` dunder method).

---


In [ ]:
import sys
import weakref

print("=" * 70)
print("SECTION 1: Object Memory Model and Identity")
print("=" * 70)

class SimpleObject:
    def __init__(self, val):
        self.val = val

# Allocate on heap
obj1 = SimpleObject(42)
obj2 = SimpleObject(42)

# Reference identity vs value equality
print(f"obj1 is obj2: {obj1 is obj2} (different heap addresses)")
print(f"obj1.val == obj2.val: {obj1.val == obj2.val}")
print(f"Memory ID obj1: {id(obj1)} | Memory ID obj2: {id(obj2)}")



---
## 2. CONSTRUCTORS AND DESTRUCTORS

### The Destructor Pipeline
`__del__(self)` is called when an object's reference count falls to zero.
- **CPython Danger**: Garbage collection via reference counting is deterministic. However, if circular references exist between objects, reference counts never reach zero. In this case, CPython's cyclic garbage collector must find and break the cycles.
- **Destructor Unpredictability**: Overriding `__del__` can prevent the garbage collector from automatically deallocating objects in older Python versions, and calling order at interpreter shutdown is not guaranteed.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: Circular References and GC")
print("=" * 70)

class Node:
    def __init__(self, name):
        self.name = name
        self.ref = None
        
    def __del__(self):
        print(f"Destructor called for Node: {self.name}")

# Create circular reference
node_a = Node("A")
node_b = Node("B")
node_a.ref = node_b
node_b.ref = node_a

# Delete variables from local scope
print("Deleting primary references...")
del node_a
del node_b
print("Nodes are NOT deleted from memory because reference count is still 1 due to circle!")

# Force collection cycle
import gc
print("\nRunning garbage collector explicitly...")
gc.collect()



---
## 3. ENCAPSULATION AND ACCESS PROTECTION

### Name Mangling
Python does not have compiler-enforced access modifiers (like `private` in C++). It uses:
- Single underscore prefix `_var`: Weak warning to developer (protected conventions).
- Double underscore prefix `__var`: Triggers name mangling. CPython automatically renames the attribute to `_ClassName__var` to prevent accidental namespace overrides in inheritance structures.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: Encapsulation & Name Mangling")
print("=" * 70)

class ProtectedClass:
    def __init__(self):
        self.public_var = 1
        self._protected_var = 2
        self.__private_var = 3  # Mangled to _ProtectedClass__private_var

obj = ProtectedClass()
print(f"Public:    {obj.public_var}")
print(f"Protected: {obj._protected_var}")

# Try private access
try:
    print(obj.__private_var)
except AttributeError as e:
    print(f"__private_var access failed: {e}")

# Access private using mangled name
print(f"Private (Mangled): {obj._ProtectedClass__private_var}")



---
## 4. INHERITANCE AND METHOD RESOLUTION ORDER (MRO)

### MRO & C3 Linearization
Python supports multiple inheritance. When resolving attributes/methods, it determines the order of traversal (MRO) using **C3 Linearization**.
C3 ensures:
- Subclasses are checked before base classes.
- The inheritance order of base classes is preserved.
- Monotonicity is maintained.

`super()` does not just call the parent class; it calls the next class in the computed MRO sequence.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: Multiple Inheritance & super() MRO")
print("=" * 70)

class A:
    def process(self):
        print("A process")

class B(A):
    def process(self):
        print("B process")
        super().process()

class C(A):
    def process(self):
        print("C process")
        super().process()

class D(B, C):
    def process(self):
        print("D process")
        super().process()

# MRO sequence for D: D -> B -> C -> A -> object
print(f"MRO order for class D:")
for cls in D.__mro__:
    print(f"  {cls}")

print("\nExecuting D.process():")
d_obj = D()
d_obj.process()



---
## 5. POLYMORPHISM AND OPERATOR OVERLOADING

Polymorphism enables different objects to respond to the same method names. Duck typing is dynamic polymorphism: "If it walks like a duck and quacks like a duck, it is a duck."

### Operator Overloading
Overloading standard operators (`+`, `-`, `*`) is done by implementing their corresponding magic methods (`__add__`, `__sub__`, `__mul__`).


In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: Operator Overloading")
print("=" * 70)

class ComplexNumber:
    def __init__(self, real, imag):
        self.real = real
        self.imag = imag

    def __add__(self, other):
        if not isinstance(other, ComplexNumber):
            return NotImplemented
        return ComplexNumber(self.real + other.real, self.imag + other.imag)

    def __repr__(self):
        return f"{self.real} + {self.imag}i"

c1 = ComplexNumber(2, 3)
c2 = ComplexNumber(4, 5)
print(f"Addition result: {c1 + c2}")



---
## 6. DUNDER METHODS REFERENCE

Dunder (Double Underscore) methods allow classes to hook into Python syntax:

| Category | Method | Description |
|:---|:---|:---|
| Lifecycle | `__new__`, `__init__`, `__del__` | Creation, Initialization, Deconstruction |
| Arithmetic | `__add__`, `__sub__`, `__mul__` | Operator overloads |
| Comparison | `__eq__`, `__lt__`, `__gt__` | Logic equality and sorting comparisons |
| Container | `__len__`, `__getitem__`, `__setitem__` | Subscripting and collection support |
| Context | `__enter__`, `__exit__` | Supporting `with` block management |
| String representation | `__str__`, `__repr__` | User-facing display, Developer debugging view |


In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: Dunder Containers and Contexts")
print("=" * 70)

class CustomContainer:
    """Mock container implementing subscription and context manager."""
    def __init__(self):
        self.data = {}

    def __setitem__(self, key, value):
        self.data[key] = value

    def __getitem__(self, key):
        return self.data.get(key, "N/A")

    def __len__(self):
        return len(self.data)

    def __enter__(self):
        print("Entering custom context block")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Exiting custom context block")
        return False

# Test container and context
with CustomContainer() as container:
    container["user"] = "Alice"
    print(f"Container data lookup: {container['user']}")
    print(f"Container length: {len(container)}")



---
## 7. DESCRIPTORS AND METACLASSES

### Descriptor Protocol
Descriptors are classes that manage access to attributes of other classes by implementing `__get__`, `__set__`, and/or `__delete__`. Properties are descriptors under the hood.

### Metaclasses
Metaclasses are the "classes of classes". They control class creation.
- The default metaclass is `type`.
- Overriding a metaclass's `__new__` or `__init__` allows you to validate or modify class definitions when they are loaded by the interpreter.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: Descriptors & Metaclasses")
print("=" * 70)

class IntValidatorDescriptor:
    """Descriptor that validates an attribute is an integer."""
    def __init__(self, name):
        self.name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.name, None)

    def __set__(self, instance, value):
        if not isinstance(value, int):
            raise TypeError(f"Attribute {self.name} must be an integer")
        setattr(instance, self.name, value)

class RegistryMetaclass(type):
    """Metaclass that keeps a registry of all classes created using it."""
    registry = {}
    
    def __new__(cls, name, bases, attrs):
        new_class = super().__new__(cls, name, bases, attrs)
        cls.registry[name] = new_class
        return new_class

class Profile(metaclass=RegistryMetaclass):
    age = IntValidatorDescriptor("age")

p = Profile()
p.age = 25
print(f"Validated Age: {p.age}")
try:
    p.age = "twenty-five"
except TypeError as e:
    print(f"Descriptor Validation Caught: {e}")

print(f"Class registry contents: {list(RegistryMetaclass.registry.keys())}")



---
## 8. DESIGN PATTERNS

### Singleton Pattern
Ensures a class has only one instance and provides a global access point.

### Factory Pattern
Defines an interface for creating objects but defers instantiation to subclasses.


In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: Design Patterns")
print("=" * 70)

class Singleton:
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

s1 = Singleton()
s2 = Singleton()
print(f"s1 is s2 (Singleton match): {s1 is s2}")

# Factory Example
class Cat:
    def speak(self): return "Meow"

class Dog:
    def speak(self): return "Woof"

class AnimalFactory:
    @staticmethod
    def get_animal(animal_type):
        if animal_type == "cat":
            return Cat()
        return Dog()

cat = AnimalFactory.get_animal("cat")
print(f"Factory animal speak: {cat.speak()}")



---
## 9. REAL-WORLD SYSTEM DESIGN USING OOP: BANKING SYSTEM

This represents a complete, multi-class object system mapping transactions, savings, and auditing rules.


In [ ]:
class InsufficientFundsError(Exception): pass

class BankAccount:
    """Manages transactional ledger entries using OOP state encapsulation."""
    def __init__(self, owner, initial_balance=0.0):
        self.owner = owner
        self._balance = initial_balance
        self.ledger = []

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self._balance += amount
        self.ledger.append(f"Deposit: +{amount}")
        return self._balance

    def withdraw(self, amount):
        if amount > self._balance:
            raise InsufficientFundsError("Insufficient funds for withdrawal")
        self._balance -= amount
        self.ledger.append(f"Withdrawal: -{amount}")
        return self._balance

print("\n=== Section 9: Banking System Simulation ===")
account = BankAccount("John Doe", 100.0)
account.deposit(50.0)
account.withdraw(30.0)
print(f"Final Balance: {account.balance} | Ledger: {account.ledger}")



---
## 10. INTERVIEW MASTER LAYER

### 100 Scenario-Based OOP Interview Q&As (Summary Selection)

1. **Explain the difference between `__new__` and `__init__`.**
   - `__new__` is a static method that allocates memory and returns the object instance. `__init__` is an instance method that initializes attributes once the instance exists.
2. **What is MRO and how does C3 linearization work?**
   - MRO determines the search order for method resolution. C3 linearization computes a list of classes by merging parent MRO structures while ensuring subclass priority and local precedence order.
3. **How does Python's Garbage Collector handle circular references?**
   - CPython uses reference counting as its primary GC. If a cycle exists, the reference counts never drop to zero. CPython's cyclic GC sweeps memory periodically, identifying isolated islands of self-referencing objects and deallocating them.
4. **What is the descriptor protocol?**
   - Descriptors are classes defining `__get__`, `__set__`, or `__delete__` magic methods, which customize how attributes are read, written, or deleted in target instances.
5. **Why should we avoid calling `__del__` manually?**
   - Deleting an object reference (`del obj`) only decrements the ref count. The destructor `__del__` is called automatically by CPython when the count hits zero. Calling `__del__` manually does not release memory; it merely executes the function code.
